# TRIBE v2 — Colab Backend
**Cell 1 installs everything and auto-restarts. After restart, run Cells 2–8.**

> Recommended: **GPU runtime** → `Runtime → Change runtime type → T4 GPU`

In [ ]:
# CELL 1 — Install (runtime auto-restarts at the end — expected)
!pip install -q uv
!uv pip install --system "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
!uv pip install --system fastapi "uvicorn[standard]" yt-dlp pydantic nest_asyncio huggingface_hub
!apt-get install -y -q ffmpeg 2>/dev/null || true
print("\n✓ Done. Restarting to flush old numpy from memory...")
import os, time; time.sleep(2)
os.kill(os.getpid(), 9)

In [ ]:
# CELL 2 — Verify (run after restart)
from tribev2.demo_utils import TribeModel
import tribev2, numpy as np, torch
print(f"tribev2 : {tribev2.__file__}")
print(f"numpy   : {np.__version__}")
print(f"torch   : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none — CPU mode'}")
print("TribeModel ✓")

In [ ]:
# CELL 3 — Fix whisperx float16 error
# tribev2 spawns whisperx as a subprocess with --compute_type float16
# float16 only works on CUDA. This cell writes a Python wrapper at
# /usr/local/bin/whisperx that intercepts the call and swaps compute_type.
import os, stat, torch

compute_type = "float16" if torch.cuda.is_available() else "int8"
print(f"compute_type → {compute_type}")

wrapper_py = f"""#!/usr/bin/env python3
# whisperx wrapper — auto-sets compute_type based on GPU availability
import sys

args = sys.argv[1:]

# Strip any existing --compute_type argument
filtered, i = [], 0
while i < len(args):
    if args[i] == '--compute_type':
        i += 2
    else:
        filtered.append(args[i])
        i += 1

# Add correct compute_type
filtered += ['--compute_type', '{compute_type}']
sys.argv = ['whisperx'] + filtered

from whisperx.__main__ import cli
sys.exit(cli())
"""

wrapper_path = "/usr/local/bin/whisperx"
with open(wrapper_path, "w") as f:
    f.write(wrapper_py)
os.chmod(wrapper_path, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)

# Verify
import subprocess
r = subprocess.run(["whisperx", "--help"], capture_output=True, text=True)
if r.returncode == 0:
    print(f"whisperx wrapper ✓  ({wrapper_path})")
else:
    print("wrapper issue:", r.stderr[:300])

In [ ]:
# CELL 4 — Clone / update brain-neuro
import os
repo = '/content/brain-neuro'
if os.path.exists(repo):
    !git -C {repo} pull -q
    print('brain-neuro updated ✓')
else:
    !git clone -q https://github.com/Sambhram1/brain-neuro.git {repo}
    print('brain-neuro cloned ✓')
os.chdir(repo)
print(f'cwd: {os.getcwd()}')

In [ ]:
# CELL 5 — HuggingFace login
import huggingface_hub
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('HF_TOKEN from Colab secrets ✓')
except Exception:
    hf_token = input('Enter HuggingFace token: ')
huggingface_hub.login(token=hf_token, add_to_git_credential=False)
print('Logged in ✓  — weights download on first /analyze call')

In [ ]:
# CELL 6 (Optional) — cookies.txt for Instagram/TikTok. Skip for YouTube.
import shutil, os
try:
    from google.colab import files
    print('Upload cookies.txt or skip this cell')
    uploaded = files.upload()
    for fname in uploaded:
        shutil.move(fname, 'cookies.txt')
        print('cookies.txt ✓')
except ImportError:
    print('cookies.txt ✓' if os.path.exists('cookies.txt') else 'No cookies.txt — YouTube only')

In [ ]:
# CELL 7 — Cloudflared tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('cloudflared ✓')

In [ ]:
# CELL 8 — Start backend + tunnel
import nest_asyncio, uvicorn, threading, subprocess, time, re, os
os.chdir('/content/brain-neuro')
nest_asyncio.apply()

def run_server():
    uvicorn.run('backend:app', host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(4)
print('FastAPI running on :8000 ✓')

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print('Waiting for tunnel...')
for line in proc.stdout:
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line.decode())
    if match:
        url = match.group()
        print(f'\n{"═"*54}')
        print(f'  BACKEND URL: {url}')
        print(f'{"═"*54}')
        print('→ Paste into the frontend Backend field')
        break